# Week 12 Lecture: The Professional ML Workflow

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/notebooks/tuesday-your-turn/week-12-lecture.ipynb)

This week you'll stop peeking at the test set. You'll use cross-validation to score models honestly, `GridSearchCV` to tune hyperparameters, and a single pipeline to impute, encode, and scale features without leaking information.

## How to use this notebook

This notebook accompanies the Week 12 Tuesday lecture. You can:

- **Follow along during class**: run each cell as the instructor discusses it
- **Work the Your Turn exercises** when prompted
- **Review after class**: rerun and experiment with the code

## Setup

Everything we need is already installed in Colab. Today's data is the cleaned **Ames housing** data, loaded straight from the course GitHub repository. Each row is a home sold in Ames, Iowa, and `SalePrice` is what we'll predict.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error
from sklearn.neighbors import KNeighborsRegressor

In [ ]:
url = "https://raw.githubusercontent.com/bradleyboehmke/uc-bana-4080/main/data/ames_clean.csv"
ames = pd.read_csv(url)

print(ames.shape)
ames.head()

---

## 1. Cross-Validation

### The peeking problem

Last week we chose settings like `max_depth` by checking the test set each time. In class, the Optimism Gap chart showed what that habit does: the best test score keeps climbing, while the real-world score of that same model climbs far less.

Here's the same habit on Ames. We split the data three ways:

- **Training set** (60%): what the trees learn from
- **Test set** (100 homes): the set we keep checking to pick a depth
- **New homes** (the rest): homes that play no part in the choice, standing in for the future

In [ ]:
features = ['GrLivArea', 'LotArea', 'YearRemodAdd', 'TotRmsAbvGrd', 'GarageArea', 'Fireplaces']
X, y = ames[features], ames['SalePrice']

X_train, X_rest, y_train, y_rest = train_test_split(X, y, train_size=0.6, random_state=0)
X_test, X_new, y_test, y_new = train_test_split(X_rest, y_rest, train_size=100, random_state=0)

print(f"Train: {len(X_train)} homes | Test: {len(X_test)} homes | New: {len(X_new)} homes")

Try 20 depths, score each one on the same 100 test homes, and keep the winner. Then score that winner on the new homes.

In [ ]:
test_rmse = {}   # try 20 depths, scoring each on the same 100 test homes
for depth in range(1, 21):
    tree = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(X_train, y_train)
    test_rmse[depth] = root_mean_squared_error(y_test, tree.predict(X_test))
best_depth = min(test_rmse, key=test_rmse.get)

# score the chosen depth on homes that played no part in the choice
best_tree = DecisionTreeRegressor(max_depth=best_depth, random_state=42).fit(X_train, y_train)
new_rmse = root_mean_squared_error(y_new, best_tree.predict(X_new))

print(f"Depth chosen by the test set:   {best_depth}")
print(f"What you report (test RMSE):    ${test_rmse[best_depth]:,.0f}")
print(f"What you get (new-home RMSE):   ${new_rmse:,.0f}")

The RMSE you would report is lower (better) than the RMSE on homes you never looked at. That's the optimism gap, on real data.

### What the test set hid

Two numbers don't show *why*. Let's score all 20 depths on both the test set and the new homes. In a real project you would only ever see the orange line.

In [ ]:
depths = range(1, 21)
new_home_rmse = [
    root_mean_squared_error(
        y_new,
        DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train).predict(X_new))
    for d in depths
]

plt.figure(figsize=(9, 4))
plt.plot(depths, [test_rmse[d] for d in depths], marker="o", color="#e58139", label="Test set (what you see)")
plt.plot(depths, new_home_rmse, marker="o", color="#399de5", label="New homes (hidden in real life)")
plt.axvline(best_depth, color="gray", linestyle=":", label=f"Depth chosen by the test set ({best_depth})")
plt.xlabel("Tree max_depth")
plt.ylabel("RMSE ($, lower is better)")
plt.xticks(list(depths))
plt.legend()
plt.show()

truly_best = min(depths, key=lambda d: new_home_rmse[d - 1])
print(f"Best depth on the new homes: {truly_best}")

Depth 9 only looked best because its errors happened to be small on those 100 test homes. Peeking inflates the score **and** can steer you to the wrong model.

> **🔒 The takeaway:** the test set can't be used to make modeling decisions. Lock it away for one job: the **final generalization assessment**, run once, after every choice is made. That leaves a gap: we still need to compare models, pick hyperparameters, and estimate performance using **only the training data**. Cross-validation fills that gap.

### K-fold cross-validation

**K-fold cross-validation** splits the **training data** into K folds (here, 5). Each round, one fold sits out as the validation set, the model trains on the other four, and it's scored on the fold that sat out. The **CV score** is the average of the K rounds. The test set never takes part.

Same Ames features, now with a depth-6 tree. Split first, lock the test set away, then cross-validate on the training set only.

In [ ]:
# Step 1: split first, then lock the test set away
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 2: 5-fold CV on the TRAINING set only
tree = DecisionTreeRegressor(max_depth=6, random_state=42)
cv_scores = cross_val_score(tree, X_train, y_train, cv=5,
                            scoring='neg_root_mean_squared_error')
cv_rmse = -cv_scores   # scikit-learn reports negative RMSE so that higher is always better

print(cv_rmse.round())   # one RMSE per fold
print(f"CV RMSE: ${cv_rmse.mean():,.0f} ± ${cv_rmse.std():,.0f}")

**Reading the output:**

- **The minus sign.** scikit-learn's scorers treat *higher as better*, so error metrics come back negative (`neg_root_mean_squared_error`). Flip the sign to get ordinary RMSE.
- **The mean** is our estimate of how a depth-6 tree will do on new homes.
- **The std** shows how much the score moves depending on which homes are held out. A big std means one validation score on its own could be misleading.

### 🎯 Your Turn: How Lucky Was Your Split?

In class we saw that a single train/test split's score depends a lot on *which* homes land in the test set. Try it yourself:

1. For each `random_state` from **0 to 9**, split `X` and `y` 80/20, fit a `DecisionTreeRegressor(max_depth=6, random_state=42)` on the training part, and print its **test RMSE**.
2. What are the lowest and highest test RMSEs you got? How does that range compare with the CV mean ± std above?
3. If you could report only one number to your manager, would you trust one split's test RMSE or the CV mean? Why?

*Hint: put the `train_test_split` call inside a `for seed in range(10):` loop and use `random_state=seed`. `root_mean_squared_error(y_true, y_pred)` gives the RMSE.*

In [ ]:
# Your code here

---

## 2. Hyperparameter Tuning with GridSearchCV

Instead of copying and pasting one `cross_val_score` call per setting, `GridSearchCV` tries **every combination** in a grid, scores each with cross-validation on the training set, and keeps track of the best.

Here's the search from class: a decision tree over 6 values of `max_depth` × 5 values of `min_samples_leaf` = 30 settings, each scored with 5-fold CV (150 models in total).

In [ ]:
param_grid = {
    'max_depth': [2, 4, 6, 8, 10, None],
    'min_samples_leaf': [1, 5, 10, 20, 40],
}
grid_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,                       # use all CPU cores
)
grid_search.fit(X_train, y_train)    # 30 settings × 5 folds = 150 models

print("Best settings:", grid_search.best_params_)
print(f"Best CV RMSE: ${-grid_search.best_score_:,.0f}")

**What you get back:**

- `best_params_`: the winning cell of the grid
- `best_score_`: its CV score (negative RMSE again, so flip the sign)
- `best_estimator_`: the winning setting **refit on all of `X_train`**, ready to test once at the very end
- `cv_results_`: the score of **every** cell, which is handy for plotting

Here are the five best cells from `cv_results_`:

In [ ]:
(
    pd.DataFrame(grid_search.cv_results_)
    .assign(cv_rmse=lambda df: -df["mean_test_score"])
    [["param_max_depth", "param_min_samples_leaf", "cv_rmse"]]
    .sort_values("cv_rmse")
    .head()
)

### Refit, then test once

CV trained 5 copies of each setting only to *score* them. Once we've chosen, we refit one model with the winning settings on **all** of the training data. (`GridSearchCV` does this for you: `grid_search.best_estimator_` is that refit model.)

Then, **only when every decision is final**, we score it on the test set, one time. That's the final generalization assessment. If it disappoints, don't re-tune against it: that's peeking again.

In [ ]:
# Step 1: refit the winning settings on ALL of the training data
final_tree = DecisionTreeRegressor(random_state=42, **grid_search.best_params_)
final_tree.fit(X_train, y_train)     # GridSearchCV already did this: grid_search.best_estimator_

# Step 2: ONLY when every decision is final, test once
test_rmse = root_mean_squared_error(y_test, final_tree.predict(X_test))
print(f"CV RMSE (our estimate):      ${-grid_search.best_score_:,.0f}")
print(f"Test RMSE (final, one time): ${test_rmse:,.0f}")

### 🎯 Your Turn: Tune a Tree

Run your own, finer grid search and plot the results.

1. Build a `GridSearchCV` for a `DecisionTreeRegressor(random_state=42)` that tries `max_depth` from **1 to 15** and `min_samples_leaf` in **[1, 5, 10, 25, 50]**. Use `cv=5`, `scoring='neg_root_mean_squared_error'`, and `n_jobs=-1`.
2. Fit it on `X_train, y_train` and print the best settings and best CV RMSE. Did they change from the search above?
3. Turn `cv_results_` into a DataFrame (add a `cv_rmse` column, as above) and plot **CV RMSE vs. `max_depth`**, with **one line per `min_samples_leaf`** value.
4. Where does the tree start to overfit? Which leaf size protects it best at large depths?

*Hint: `range(1, 16)` gives depths 1–15. For the plot, `results.pivot(index="param_max_depth", columns="param_min_samples_leaf", values="cv_rmse").plot(marker="o")` draws one line per leaf size.*

In [ ]:
# Your code here

---

## 3. Feature Engineering & Pipelines

**Feature engineering** is turning raw columns into inputs a model can learn from. You've already done it: in weeks 10–11 we converted Yes/No columns to 1/0 with `(df["col"] == "Yes").astype(int)`. That works, but it lives outside the model, so new data has to be converted the exact same way, and a surprise value like `"yes"` silently becomes 0. scikit-learn's encoders inside a pipeline fix both.

Today's two most common moves:

- **Scale numeric features** so distance-based models like KNN weigh them equally (`StandardScaler`: each value becomes standard deviations from its column's mean)
- **Encode categorical features**: **one-hot** for categories with no order, **ordinal** for categories with a real order

### One-hot encoding

For categories with **no natural order**, like house style. Three actual homes: each gets a 1 in its own style's column and 0 everywhere else.

In [ ]:
homes = ames.loc[[1, 0, 275], ["Id", "HouseStyle"]].set_index("Id")   # three actual homes
one_hot = OneHotEncoder(sparse_output=False)
encoded = one_hot.fit_transform(homes)

homes.join(pd.DataFrame(encoded, index=homes.index,
                        columns=one_hot.get_feature_names_out()).astype(int))

### Ordinal encoding

For categories with a **real order**, like kitchen quality: Fair < Typical < Good < Excellent. One column in, one column out, and the numbers keep the order.

> ⚠️ Always pass `categories=`. Without it, scikit-learn sorts the levels alphabetically (Ex = 0, Fa = 1, Gd = 2, TA = 3), which scrambles the order.

In [ ]:
quality_order = ["Fa", "TA", "Gd", "Ex"]   # Fair < Typical < Good < Excellent
homes = ames.loc[[1, 0, 275], ["Id", "KitchenQual"]].set_index("Id")
ordinal = OrdinalEncoder(categories=[quality_order])

homes.assign(KitchenQual_encoded=ordinal.fit_transform(homes).ravel().astype(int))

### Feature engineering inside a pipeline

A `ColumnTransformer` gives each group of columns its own treatment: scale the numeric columns, one-hot encode the unordered categories, ordinal-encode the quality ratings. A `Pipeline` glues that preprocessing to a KNN model, so `cross_val_score` refits **everything** in each fold. The scaler and encoders only ever learn from that fold's training rows, so nothing leaks from the validation fold or the test set.

In [ ]:
numeric = ["GrLivArea", "LotArea", "YearRemodAdd", "TotRmsAbvGrd", "GarageArea", "Fireplaces"]
categorical = ["HouseStyle", "MSZoning"]
quality = ["KitchenQual", "ExterQual"]

preprocess = ColumnTransformer([
    ("scale", StandardScaler(), numeric),                                   # numbers → scaled
    ("one_hot", OneHotEncoder(handle_unknown="ignore"), categorical),       # no order → 0/1 columns
    ("ordinal", OrdinalEncoder(categories=[quality_order] * 2), quality),   # real order → 0, 1, 2, 3
])
knn_pipeline = Pipeline([("prep", preprocess),
                         ("model", KNeighborsRegressor(n_neighbors=10))])

X_train, X_test, y_train, y_test = train_test_split(
    ames[numeric + categorical + quality], ames["SalePrice"], test_size=0.2, random_state=42)
cv_rmse = -cross_val_score(knn_pipeline, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error")
print(f"CV RMSE: ${cv_rmse.mean():,.0f}")

**Does scaling matter?** Swap the scaler for `"passthrough"` (leave the numbers as they are) and compare:

In [ ]:
unscaled = ColumnTransformer([
    ("raw", "passthrough", numeric),
    ("one_hot", OneHotEncoder(handle_unknown="ignore"), categorical),
    ("ordinal", OrdinalEncoder(categories=[quality_order] * 2), quality),
])
unscaled_pipeline = Pipeline([("prep", unscaled),
                              ("model", KNeighborsRegressor(n_neighbors=10))])
cv_rmse_raw = -cross_val_score(unscaled_pipeline, X_train, y_train, cv=5,
                               scoring="neg_root_mean_squared_error")
print(f"CV RMSE without scaling: ${cv_rmse_raw.mean():,.0f}")

### Tuning the whole pipeline

A pipeline acts like a model, so `GridSearchCV` can tune it. To reach a setting *inside* a step, write **the step's name**, **two underscores**, then **the parameter**: `model__n_neighbors` means "the `n_neighbors` of the step we named `model`". (`knn_pipeline.get_params().keys()` lists every name you can use.)

In [ ]:
param_grid = {"model__n_neighbors": [3, 5, 7, 10, 15, 25]}   # step name + __ + parameter
pipe_search = GridSearchCV(knn_pipeline, param_grid, cv=5,
                           scoring="neg_root_mean_squared_error", n_jobs=-1)
pipe_search.fit(X_train, y_train)
print("Best settings:", pipe_search.best_params_)
print(f"Best CV RMSE: ${-pipe_search.best_score_:,.0f}")

> **📚 There's a lot more to feature engineering.** Imputing missing values, creating new features and more are in this week's reading (Chapter 30). They all plug into the same `ColumnTransformer`.

---

## 🧾 What You Learned

| Code | What it does |
|------|--------------|
| `cross_val_score(model, X, y, cv=5, scoring=...)` | Score a model on 5 train/validation splits of the training data |
| `GridSearchCV(model, param_grid, cv=5)` | Try every combination in a grid, scoring each with cross-validation |
| `StandardScaler()` | Scale numeric features to mean 0, std 1 |
| `OneHotEncoder(handle_unknown="ignore")` | Turn each unordered category into its own 0/1 column |
| `OrdinalEncoder(categories=[order])` | Turn ordered categories into 0, 1, 2, ... in the order you give |
| `ColumnTransformer([...])` | Apply different preprocessing to different columns |
| `Pipeline([...])` | Chain preprocessing and a model so each CV fold fits them on its own training data |